# App RAG empresarial con NVIDIA NIM — mi copia (Bloque A)

Copia personal del notebook `lecture10/02b`. Incluye los cambios de las tareas del Bloque A:

- **A2** — carga de mi corpus propio (`data/company_documents.json`, Club de Montañismo Cumbre Andina) y `chunk_size` ajustado.
- **A3** — nuevo *system prompt* que obliga a citar el título entre corchetes, usar una frase fija cuando falta contexto y responder en 3 oraciones máximo.
- **A4** — `build_pipeline` devuelve también el `vectorstore` y la interfaz muestra el *score* de similitud de cada fuente con `similarity_search_with_score`.

**Modelos:** embeddings `nvidia/nemotron-3-embed-1b`, generación `nvidia/nemotron-3-super-120b-a12b`.

> Ejecuta las celdas en orden en Colab. La última celda publica una URL pública de Gradio.

> **Ruta del corpus (A2):** este notebook carga `Path("data/company_documents.json")`. Sube el archivo dentro de una carpeta `data/` en la sesión de Colab, o clona el repo. Si prefieres subirlo a la raíz, cambia la ruta a `Path("company_documents.json")`.

## 1. Instalar dependencias

In [ ]:
%pip install -qU gradio langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv


## 2. Configurar la API key

Crea el secreto `NVIDIA_API_KEY` en el panel 🔑 Secrets de Colab y habilita el acceso para este notebook.

In [ ]:
import os
import json
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


## 3. Cargar los documentos (A2 — mi corpus)

Reemplazo los 4 documentos de demostración por mi corpus del **Club de Montañismo Cumbre Andina** (9 documentos, cada uno con `department` y `type` en `metadata`).

In [ ]:
DEMO_DOCUMENTS = [
    {"id": "product_001", "title": "CloudSync Pro — Precios",
     "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual.",
     "metadata": {"department": "sales", "type": "pricing"}},
]


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS
    with path.open(encoding="utf-8") as file:
        documents = json.load(file)
    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


# A2: ruta corregida respecto al notebook original. El archivo vive en data/.
company_documents = load_documents(Path("data/company_documents.json"))
print(f"✅ {len(company_documents)} documentos disponibles")


## 4. Construir el pipeline RAG (A3 prompt + A4 devuelve vectorstore)

`chunk_size=600`, `chunk_overlap=80`: mis documentos son párrafos cortos y autocontenidos; con 400 se partían frases a la mitad y se perdía el precio o el umbral asociado a cada política. 600 mantiene cada política en un solo chunk sin diluir la señal del embedding.

In [ ]:
def build_pipeline(documents: List[Dict]):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(model=GENERATION_MODEL, temperature=0.2, top_p=0.95, max_completion_tokens=700)

    # A2: chunk_size ajustado de 400 -> 600 para documentos cortos.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=80,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )

    # ===== A3: nuevo system prompt con intención =====
    prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            "Eres el asistente del Club de Montañismo Cumbre Andina. Responde únicamente "
            "con información del contexto. Reglas estrictas:\n"
            "1. Cita SIEMPRE el título del documento usado entre corchetes, por ejemplo [Planes de membresía].\n"
            "2. Si el contexto no contiene la respuesta, responde EXACTAMENTE: "
            "'No tengo esa información en los documentos del club.'\n"
            "3. Máximo 3 oraciones. Responde en español.",
        ),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    # A4: devolver también el vectorstore para poder mostrar scores.
    return pipeline, chunks, vectorstore


print("Construyendo el índice...")
rag_pipeline, indexed_chunks, vectorstore = build_pipeline(company_documents)
titles = [document["title"] for document in company_documents]
print(f"✅ Pipeline listo: {len(titles)} documentos, {len(indexed_chunks)} chunks")


## 5. Interfaz Gradio con scores (A4)

`chat()` usa `vectorstore.similarity_search_with_score(question, k=3)` para mostrar cada fuente con su score de similitud (coseno; mayor = más parecido).

In [ ]:
def chat(question: str, history: list | None):
    history = history or []
    question = (question or "").strip()
    if not question:
        return "", history, "Escribe una pregunta."

    try:
        result = rag_pipeline.invoke(question)
        answer = result["answer"].strip()

        # A4: scores por fuente.
        scored = vectorstore.similarity_search_with_score(question, k=3)
        if scored:
            lineas = []
            for doc, score in scored:
                title = doc.metadata.get("title", "Sin título")
                lineas.append(f"- **{title}** · score = `{score:.3f}`")
            sources_md = "**Fuentes recuperadas (con score):**\n" + "\n".join(lineas)
        else:
            sources_md = "No se encontraron documentos relevantes."
    except Exception as error:
        answer = "Ocurrió un error al consultar el pipeline."
        sources_md = f"**Error:** `{type(error).__name__}: {error}`"

    history = history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return "", history, sources_md


def clear_chat():
    return [], ""


with gr.Blocks(title="RAG Club Cumbre Andina") as demo:
    gr.Markdown("# RAG · Club de Montañismo Cumbre Andina\nLangChain + NVIDIA NIM + Qdrant. Las fuentes muestran su score de similitud.")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Documentos indexados")
            gr.Markdown("\n".join(f"- {title}" for title in titles))
            gr.Markdown(f"*{len(indexed_chunks)} chunks en el índice*")
            gr.Markdown(f"**Embeddings:** `{EMBEDDING_MODEL}`  \n**LLM:** `{GENERATION_MODEL}`  \n**Retrieval:** k=3, threshold=0.3")
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=430, label="Chat")
            with gr.Row():
                question_box = gr.Textbox(placeholder="Pregunta sobre el club...", label="Pregunta", lines=2, scale=4)
                send_button = gr.Button("Enviar", variant="primary", scale=1)
            sources_box = gr.Markdown()
            clear_button = gr.Button("Limpiar", variant="secondary")

    gr.Examples(
        examples=[
            ["¿Cuánto cuesta el plan Expedición?"],
            ["¿Cuántos guías certificados se necesitan por participante?"],
            ["¿Qué requisitos hay para una expedición sobre 4.500 metros?"],
            ["¿Cómo funciona el descuento por voluntariado?"],
        ],
        inputs=question_box,
        label="Preguntas de ejemplo",
    )
    send_button.click(chat, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
    question_box.submit(chat, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
    clear_button.click(clear_chat, outputs=[chatbot, sources_box])


## 6. Ejecutar la app

In [ ]:
demo.queue().launch(share=True, debug=True, show_error=True)
